# 🔬 Lifelong Image Retrieval using DwoPP & Episodic Metric Learning (Kaggle 2-GPU)
Notebook này hướng dẫn chi tiết cách chạy huấn luyện tăng trưởng và đánh giá hệ thống truy xuất ảnh suốt đời (**Lifelong Image Retrieval**) cho cả **4 Tasks** tuần tự trên Kaggle.

### ⚙️ Thiết kế mô hình & hàm Loss mới (DwoPP):
1. **Retrieval Projection Head**: Một lớp chiếu Conv2D cục bộ được tích hợp vào Decoupled Head của YOLO-World để ánh xạ đặc trưng vùng về 256 chiều.
2. **Episodic Hard-mining Metric Loss ($L_{eps}$)**: Triplet Loss áp dụng Batch-Hard mining để tối ưu hóa khoảng cách Euclidean của các mẫu dương cục bộ.
3. **Distillation without Positive Pairs ($L_{DwoPP}$)**: Hàm loss chưng cất tri thức từ mô hình nhiệm vụ trước nhưng loại bỏ hoàn toàn lớp tích cực (positive class) khỏi phân phối xác suất nhằm bảo toàn không gian metric mà không bị quên lãng thảm họa.
4. **Text Projection Layer**: Ánh xạ class embeddings văn bản từ 512 chiều về 256 chiều để đối sánh trực tiếp với đặc trưng vùng ảnh.

### ⚙️ Hỗ trợ Checkpoint Pre-trained:
* Nếu bạn đã pre-train trước các checkpoint phát hiện đối tượng (Object Detection) gốc của mô hình, bạn có thể cấu hình để nạp thẳng các checkpoint đó tại mỗi Task để học căn chỉnh không gian metric một cách nhanh chóng.

### ⚠️ Yêu cầu trước khi chạy:
1. Hãy chắc chắn rằng bạn đã kích hoạt **GPU T4 x2** trong phần settings của Kaggle (*Accelerator -> GPU T4 x2*).
2. Bật kết nối internet cho notebook (*Internet on*).

## 🛠️ Bước 1: Clone Repository & Submodules

In [1]:
import os
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("-> Đang clone repository từ GitHub...")
    !git clone {repo_url} {working_dir}
else:
    print("-> Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# Tải mmyolo vào thư mục third_party nếu chưa có
if not os.path.exists("third_party/mmyolo"):
    print("-> Đang tải submodule mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("-> Submodule mmyolo đã có sẵn.")

-> Đang clone repository từ GitHub...
Cloning into '/kaggle/working/OW_OVD'...
remote: Enumerating objects: 1385, done.
remote: Counting objects: 100% (425/425), done.
remote: Compressing objects: 100% (300/300), done.
remote: Total 1385 (delta 301), reused 239 (delta 122), pack-reused 960 (from 1)
Receiving objects: 100% (1385/1385), 2.62 MiB | 9.91 MiB/s, done.
Resolving deltas: 100% (942/942), done.
/kaggle/working/OW_OVD
-> Đang tải submodule mmyolo...
Cloning into 'third_party/mmyolo'...
remote: Enumerating objects: 4968, done.
remote: Counting objects: 100% (1341/1341), done.
remote: Compressing objects: 100% (294/294), done.
remote: Total 4968 (delta 1133), reused 1047 (delta 1047), pack-reused 3627 (from 1)
Receiving objects: 100% (4968/4968), 3.62 MiB | 10.62 MiB/s, done.
Resolving deltas: 100% (3216/3216), done.


In [2]:
# Thay thế lệnh git pull cũ bằng cụm này:
!git fetch --all
!git reset --hard origin/master

Fetching origin
HEAD is now at a5e8129 Disable transformers progress bar to fix Kaggle stdout lock


## 📦 Bước 2: Cài đặt Dependencies & Vá lỗi MMCV

In [3]:
print("-> 1. Thiết lập phiên bản PyTorch & Torchvision...")
!pip install -q torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

print("\n-> 2. Cài đặt MMCV từ wheel index...")
!pip install -q mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

print("\n-> 3. Cài đặt các thư viện bổ trợ...")
!pip install -q matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

print("\n-> 4. Cài đặt MMDetection...")
!pip install -q "mmdet>=3.1.0" --no-deps

print("\n-> 5. Cài đặt MMYOLO từ source...")
!pip install -q --no-build-isolation --no-deps third_party/mmyolo

print("\n-> 6. Vá lỗi kiểm tra phiên bản MMCV vật lý trên đĩa cứng...")
import site
import os
import glob
import shutil

def patch_file(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        new_content = content
        for old_ver in ["'2.1.0'", "'2.2.0'", '"2.1.0"', '"2.2.0"']:
            new_content = new_content.replace(f"mmcv_maximum_version = {old_ver}", "mmcv_maximum_version = '2.3.0'")
        if new_content != content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            print(f"  [Vá lỗi] Đã cập nhật file: {file_path}")

def clear_pycache(root_dir):
    if not os.path.exists(root_dir):
        return
    for root, dirs, files in os.walk(root_dir):
        for d in dirs:
            if d == "__pycache__":
                pycache_path = os.path.join(root, d)
                try:
                    shutil.rmtree(pycache_path)
                except Exception:
                    pass

site_dirs = site.getsitepackages()
for s_dir in site_dirs:
    for pkg in ["mmdet", "mmyolo"]:
        pkg_dir = os.path.join(s_dir, pkg)
        patch_file(os.path.join(pkg_dir, "__init__.py"))
        clear_pycache(pkg_dir)

for init_file in glob.glob("**/mmyolo/__init__.py", recursive=True):
    patch_file(init_file)
    clear_pycache(os.path.dirname(init_file))
for init_file in glob.glob("**/mmdet/__init__.py", recursive=True):
    patch_file(init_file)
    clear_pycache(os.path.dirname(init_file))

paths_to_glob = [
    "/opt/conda/lib/python*/site-packages/mmdet/__init__.py",
    "/opt/conda/lib/python*/site-packages/mmyolo/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmdet/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmyolo/__init__.py"
]
for path_pattern in paths_to_glob:
    for init_file in glob.glob(path_pattern):
        patch_file(init_file)
        clear_pycache(os.path.dirname(init_file))

print("\n-> 7. Kiểm tra import tất cả các package...")
import torch
import mmcv

real_mmcv_version = mmcv.__version__
mmcv.__version__ = '2.0.1'

import mmdet
import mmyolo
mmcv.__version__ = real_mmcv_version

print(f"  - torch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"  - mmcv: {mmcv.__version__}")
print(f"  - mmdet: {mmdet.__version__}")
print(f"  - mmyolo: {mmyolo.__version__}")
print("====== Khởi tạo môi trường hoàn tất! ======")

-> 1. Thiết lập phiên bản PyTorch & Torchvision...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 84.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 121.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 203.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 158.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 74.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 118.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 43.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 90.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 41.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 8

## 🗂️ Bước 3: Định vị Dataset & Sinh Đặc trưng nhãn bằng CLIP

In [4]:
import json
import torch
import numpy as np
import os
import glob
from transformers import AutoTokenizer, CLIPTextModelWithProjection

# Đè hàm kiểm tra bảo mật PyTorch 2.6 của transformers
import transformers
transformers.utils.import_utils.check_torch_load_is_safe = lambda *args, **kwargs: None
transformers.utils.check_torch_load_is_safe = lambda *args, **kwargs: None
transformers.modeling_utils.check_torch_load_is_safe = lambda *args, **kwargs: None

os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

weights_path = '/kaggle/input/models/nta212/yolo-world/pytorch/default/1/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    print("-> Đang tải pretrained weights của YOLO-World...")
    # !wget -O {weights_path} https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth

dataset_root = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-for-object-detection',
    'data/IP102',
    '.'
]:
    if os.path.exists(os.path.join(path, 'train.json')):
        dataset_root = path
        break
if dataset_root is None:
    paths = glob.glob('/kaggle/input/**/train.json', recursive=True)
    if paths:
        dataset_root = os.path.dirname(paths[0])

print(f"-> Thư mục Dataset IP102: {dataset_root}")
class_names = [str(i) for i in range(102)]

class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

print("-> Đang sinh text embeddings bằng CLIP...")
model_name = '/kaggle/input/models/yujkaggle/openaiclip-vit-base-patch32/pytorch/default/1'
tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
clip_model = CLIPTextModelWithProjection.from_pretrained(model_name, local_files_only=True)
clip_model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = clip_model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))

num_att = len(class_names) * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')

thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')
print("====== Khởi tạo và sinh đặc trưng nhãn hoàn tất! ======")

-> Đang tải pretrained weights của YOLO-World...
-> Thư mục Dataset IP102: /kaggle/input/datasets/nta212/ip102-for-object-detection
-> Đang sinh text embeddings bằng CLIP...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: /kaggle/input/models/yujkaggle/openaiclip-vit-base-patch32/pytorch/default/1
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight  

====== Khởi tạo và sinh đặc trưng nhãn hoàn tất! ======


## ⚙️ Bước 4: Khai báo Checkpoint Phát hiện có sẵn (Pre-trained Detection Checkpoints)
Nếu bạn đã huấn luyện trước các checkpoint phát hiện đối tượng gốc và muốn dùng checkpoint đó làm khởi tạo để chỉ tập trung tối ưu không gian metric truy xuất:
* Hãy điền đường dẫn checkpoint vào biến `PRETRAINED_DET_CHECKPOINTS` bên dưới.
* Nếu không có, hãy giữ giá trị `None` để hệ thống tự động học nối tiếp từ đầu.

In [5]:
PRETRAINED_DET_CHECKPOINTS = {
    "task_1": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/best_coco_Current class AP50_epoch_5.pth",
    "task_2": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t2.pth",
    "task_3": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t3.pth",
    "task_4": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t4.pth"
}

print("-> Đã khai báo cấu hình checkpoints ban đầu.")

-> Đã khai báo cấu hình checkpoints ban đầu.


## 🛠️ Hàm bổ trợ ghi đè cấu hình để tránh lỗi khoảng trắng trong CLI
Việc ghi đè trực tiếp `load_from` vào file cấu hình Python sẽ tránh hoàn toàn các lỗi parser của MMEngine đối với tên file chứa khoảng trắng.

In [6]:
def prepare_config_with_checkpoint(task_idx, init_checkpoint):
    config_path = f"NewRetrieval_02/ip102_t{task_idx}_retrieval.py"
    with open(config_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Lọc bỏ dòng load_from cũ nếu có
    new_lines = [line for line in lines if not line.strip().startswith('load_from')]
    
    # Thêm khai báo load_from trực tiếp vào cuối file
    if init_checkpoint is not None:
        new_lines.append(f'\nload_from = {repr(init_checkpoint)}\n')
        
    with open(config_path, 'w', encoding='utf-8') as f:
        f.writelines(new_lines)
    print(f"-> Cấu hình {config_path} đã được cập nhật load_from = {init_checkpoint}")

## 🚀 Bước 5: Huấn luyện & Đánh giá Nhiệm vụ 1 (Task 1 - 7 Lớp đầu)
Huấn luyện khớp không gian metric của 7 lớp đầu tiên.

In [7]:
import os; os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t1_retrieval.py"
checkpoint_save_dir = "work_dirs/ip102_t1_retrieval"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_1"]
if init_checkpoint is None:
    init_checkpoint = "/kaggle/input/models/nta212/yolo-world/pytorch/default/1/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth"

# Ghi checkpoint tĩnh trực tiếp vào file config
prepare_config_with_checkpoint(1, init_checkpoint)

print(f"-> Bắt đầu huấn luyện Task 1...")
os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29500",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch"
]
subprocess.run(cmd, check=True)

-> Cấu hình NewRetrieval_02/ip102_t1_retrieval.py đã được cập nhật load_from = /kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/best_coco_Current class AP50_epoch_5.pth
-> Bắt đầu huấn luyện Task 1...


W0823 15:43:43.297000 135164152140928 torch/distributed/run.py:779] 
W0823 15:43:43.297000 135164152140928 torch/distributed/run.py:779] *****************************************
W0823 15:43:43.297000 135164152140928 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0823 15:43:43.297000 135164152140928 torch/distributed/run.py:779] *****************************************
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` 

08/23 15:45:34 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
08/23 15:45:34 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
08/23 15:45:35 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False

/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  atts = torch.load(att_embeddings)
/kaggle/working/OW_

08/23 15:45:36 - mmengine - INFO - Using SyncBatchNorm()
08/23 15:45:36 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(9           ) YOLOv5ParamSchedulerHook           
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
(NORMAL      ) PipelineSwitchHook                 
(NORMAL      ) OurWorkPiplineHook                 
 -------------------- 
before_train_iter:
(9           ) YOLOv5ParamSchedulerHook           
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      

...

Loads checkpoint by local backend from path: /kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/best_coco_Current class AP50_epoch_5.pth
[OurHeadRetrieval] bbox_head.text_projection.weight not found in state_dict. Initializing text_projection randomly.
[OurHead] Dynamically resizing self.att_embeddings from torch.Size([2550, 512]) to torch.Size([175, 512]) to match checkpoint.
[OurHeadRetrieval] bbox_head.text_projection.weight not found in state_dict. Initializing text_projection randomly.
[OurHead] Dynamically resizing self.att_embeddings from torch.Size([2550, 512]) to torch.Size([175, 512]) to match checkpoint.
The model and loaded state dict do not match exactly

size mismatch for embeddings: copying a param with shape torch.Size([25, 512]) from checkpoint, the shape in current model is torch.Size([102, 512]).
missing keys in source state_dict: bbox_head.head_module.ret_preds.0.0.conv.weight, bbox_head.head_module.ret_preds.0.0.bn.weight, bbox_head.head_module.re

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()
/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()
/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/23 15:48:27 - mmengine - INFO - Exp name: ip102_t1_retrieval_20260823_154533
08/23 15:48:27 - mmengine - INFO - Epoch(train) [1][48/48]  base_lr: 1.0000e-04 lr: 4.7000e-06  eta: 0:00:00  time: 3.4616  data_time: 0.0470  memory: 14164  grad_norm: nan  loss: 283.5729  loss_cls: 128.2854  loss_bbox: 65.7915  loss_dfl: 88.8072  loss_retrieval: 0.6889  loss_dwopp: 0.0000
thr: 0.55
thr: 0.55
Saved collected distributions to data/IP102/mowod_distribution_sim1.pthSaved collected distributions to data/IP102/mowod_distribution_sim1.pth

Selected 175 attributes. Saved to data/IP102/selected_att_embeddings.pth
disable log
08/23 15:48:27 - mmengine - INFO - Saving checkpoint at 1 epochs
Selected 175 attributes. Saved to data/IP102/selected_att_embeddings.pth
disable log
08/23 15:48:27 - mmengine - WARNING - `save_param_scheduler` is True but `self.param_schedulers` is None, so skip saving parameter schedulers
08/23 15:50:03 - mmengine - INFO - Evaluating voc_2007_test using 2012 metric. Note tha

[rank0]:[W823 15:50:50.947758733 ProcessGroupNCCL.cpp:1168] Warning: WARNING: process group has NOT been destroyed before we destruct ProcessGroupNCCL. On normal program exit, the application should call destroy_process_group to ensure that any pending NCCL operations have finished in this process. In rare cases this process can exit before this point and block the progress of another member of the process group. This constraint has always been present,  but this warning has only been added since PyTorch 2.4 (function operator())


CompletedProcess(args=['torchrun', '--nproc_per_node=2', '--master_port=29500', 'third_party/mmyolo/tools/train.py', 'NewRetrieval_02/ip102_t1_retrieval.py', '--launcher', 'pytorch'], returncode=0)

In [8]:
import os; os.environ.pop('CUDA_VISIBLE_DEVICES', None)
# Force kill any orphaned training/distributed processes to release GPU resources and locks
import subprocess
subprocess.run("pkill -f train.py", shell=True)
subprocess.run("pkill -f torchrun", shell=True)
import time
time.sleep(2)  # Give the system a moment to release GPU resources
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 1...")
best_checkpoint = "work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cmd = [
    "python", "-u",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t1_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "1",
    "--query-cache", "query_cache_t1.pkl",
    "--gallery-cache", "gallery_cache_t1.pkl",
    "--output-report", "retrieval_lifelong_report_t1.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

-> Đang thực hiện đánh giá suốt đời sau Task 1...
-> Fully patched transformers check_torch_load_is_safe across namespaces


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


-> Copying CLIP model from /kaggle/input/models/yujkaggle/openaiclip-vit-base-patch32/pytorch/default/1 to /tmp/clip_model to prevent mmap hang...
      LIFELONG IMAGE RETRIEVAL EVALUATION PIPELINE      
-> Reading annotations...
-> Found 2176 query images and 2713 gallery images.
-> Extracting Query embeddings...


/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  atts = torch.load(att_embeddings)
/usr/local/lib/pyth

Loads checkpoint by local backend from path: work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth
[OurHead] Dynamically resizing self.att_embeddings from torch.Size([2550, 512]) to torch.Size([175, 512]) to match checkpoint.
-> Loading CLIP model: /tmp/clip_model


CLIPModel LOAD REPORT from: /tmp/clip_model
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Query Extraction (BBox Detection):   0%|          | 0/2176 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/Tensor

-> Extracting Gallery embeddings...


Matching Queries: 100%|██████████| 2176/2176 [00:02<00:00, 808.53it/s]


-> Saved ROC Curve plot to: roc_curve_task_1.png

======================================== EVALUATION SUMMARY Task 1 ========================================
Global mAP:       0.2138
Recall@1:         0.5550
Recall@5:         0.8023
Recall@10:        0.8880
Recall@1 (Seen):  0.7302
Recall@1 (Unseen):0.6162
OOD AUROC:        0.7134
OOD FPR@TPR95:    0.7302
----------------------------------------
Plasticity:       0.2857
Forgetting (mAP): 0.0000 (0.00%)
Overall Change:   0.2857

-> Saved lifelong markdown evaluation report to: retrieval_lifelong_report_t1.md


CompletedProcess(args=['python', '-u', 'NewRetrieval_02/evaluate_retrieval_lifelong.py', '--config', 'NewRetrieval_02/ip102_t1_retrieval.py', '--checkpoint', 'work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth', '--dataset-root', '/kaggle/input/datasets/nta212/ip102-for-object-detection', '--current-task', '1', '--query-cache', 'query_cache_t1.pkl', '--gallery-cache', 'gallery_cache_t1.pkl', '--output-report', 'retrieval_lifelong_report_t1.md', '--history-file', 'history_metrics.json'], returncode=0)

## 🚀 Bước 6: Huấn luyện & Đánh giá Nhiệm vụ 2 (Task 2 - Thêm 6 lớp mới là 13 Lớp)
Nạp checkpoint học được từ Task 1 (hoặc checkpoint pretrain của Task 2 nếu khai báo) để tiếp tục huấn luyện.

In [9]:
import os; os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t2_retrieval.py"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_2"]
if init_checkpoint is None:
    init_checkpoint = "work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth"

prepare_config_with_checkpoint(2, init_checkpoint)

print(f"-> Bắt đầu huấn luyện Task 2...")
os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29501",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch"
]
subprocess.run(cmd, check=True)

-> Cấu hình NewRetrieval_02/ip102_t2_retrieval.py đã được cập nhật load_from = /kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t2.pth
-> Bắt đầu huấn luyện Task 2...


W0823 16:01:07.730000 135876154975360 torch/distributed/run.py:779] 
W0823 16:01:07.730000 135876154975360 torch/distributed/run.py:779] *****************************************
W0823 16:01:07.730000 135876154975360 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0823 16:01:07.730000 135876154975360 torch/distributed/run.py:779] *****************************************
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` 

08/23 16:02:16 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
08/23 16:02:16 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
08/23 16:02:17 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False

/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  atts = torch.load(att_embeddings)
/kaggle/working/OW_

08/23 16:02:18 - mmengine - INFO - Using SyncBatchNorm()
08/23 16:02:18 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(9           ) YOLOv5ParamSchedulerHook           
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
(NORMAL      ) PipelineSwitchHook                 
(NORMAL      ) OurWorkPiplineHook                 
 -------------------- 
before_train_iter:
(9           ) YOLOv5ParamSchedulerHook           
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      

...

[OurHeadRetrieval] Syncing weights to old_head_module for DwoPP distillation.[OurHeadRetrieval] Syncing weights to old_head_module for DwoPP distillation.



/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/23 16:08:35 - mmengine - INFO - Epoch(train) [1][50/72]  base_lr: 1.0000e-04 lr: 4.9000e-06  eta: 0:02:43  time: 7.4438  data_time: 0.0501  memory: 13883  grad_norm: nan  loss: 287.2652  loss_cls: 142.3808  loss_bbox: 60.3626  loss_dfl: 83.8378  loss_retrieval: 0.6828  loss_dwopp: 0.0012
thr: 0.55
08/23 16:11:21 - mmengine - INFO - Exp name: ip102_t2_retrieval_20260823_160216
thr: 0.55
Saved collected distributions to data/IP102/mowod_distribution_sim1.pth
Saved collected distributions to data/IP102/mowod_distribution_sim1.pth
Selected 325 attributes. Saved to data/IP102/selected_att_embeddings.pth
disable log
Selected 325 attributes. Saved to data/IP102/selected_att_embeddings.pth
disable log
08/23 16:11:21 - mmengine - INFO - Saving checkpoint at 1 epochs
08/23 16:11:22 - mmengine - WARNING - `save_param_scheduler` is True but `self.param_schedulers` is None, so skip saving parameter schedulers
08/23 16:13:01 - mmengine - INFO - Evaluating voc_2007_test using 2012 metric. Note tha

[rank0]:[W823 16:13:49.844677174 ProcessGroupNCCL.cpp:1168] Warning: WARNING: process group has NOT been destroyed before we destruct ProcessGroupNCCL. On normal program exit, the application should call destroy_process_group to ensure that any pending NCCL operations have finished in this process. In rare cases this process can exit before this point and block the progress of another member of the process group. This constraint has always been present,  but this warning has only been added since PyTorch 2.4 (function operator())


CompletedProcess(args=['torchrun', '--nproc_per_node=2', '--master_port=29501', 'third_party/mmyolo/tools/train.py', 'NewRetrieval_02/ip102_t2_retrieval.py', '--launcher', 'pytorch'], returncode=0)

In [10]:
import os; os.environ.pop('CUDA_VISIBLE_DEVICES', None)
# Force kill any orphaned training/distributed processes to release GPU resources and locks
import subprocess
subprocess.run("pkill -f train.py", shell=True)
subprocess.run("pkill -f torchrun", shell=True)
import time
time.sleep(2)  # Give the system a moment to release GPU resources
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 2...")
best_checkpoint = "work_dirs/ip102_t2_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cmd = [
    "python", "-u",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t2_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "2",
    "--query-cache", "query_cache_t2.pkl",
    "--gallery-cache", "gallery_cache_t2.pkl",
    "--output-report", "retrieval_lifelong_report_t2.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

-> Đang thực hiện đánh giá suốt đời sau Task 2...
-> Fully patched transformers check_torch_load_is_safe across namespaces


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


      LIFELONG IMAGE RETRIEVAL EVALUATION PIPELINE      
-> Reading annotations...
-> Found 2176 query images and 2713 gallery images.
-> Extracting Query embeddings...


/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  atts = torch.load(att_embeddings)
/usr/local/lib/pyth

Loads checkpoint by local backend from path: work_dirs/ip102_t2_retrieval/best_coco_Current class AP50_epoch_1.pth
[OurHead] Dynamically resizing self.att_embeddings from torch.Size([2550, 512]) to torch.Size([325, 512]) to match checkpoint.
-> Loading CLIP model: /tmp/clip_model


CLIPModel LOAD REPORT from: /tmp/clip_model
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Query Extraction (BBox Detection):   0%|          | 0/2176 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/Tensor

-> Extracting Gallery embeddings...


Matching Queries: 100%|██████████| 2176/2176 [00:02<00:00, 836.03it/s]


-> Saved ROC Curve plot to: roc_curve_task_2.png

======================================== EVALUATION SUMMARY Task 2 ========================================
Global mAP:       0.2142
Recall@1:         0.5547
Recall@5:         0.8031
Recall@10:        0.8874
Recall@1 (Seen):  0.6171
Recall@1 (Unseen):0.6704
OOD AUROC:        0.6710
OOD FPR@TPR95:    0.7336
----------------------------------------
Plasticity:       0.1271
Forgetting (mAP): 0.0000 (0.00%)
Overall Change:   0.1271

-> Saved lifelong markdown evaluation report to: retrieval_lifelong_report_t2.md


CompletedProcess(args=['python', '-u', 'NewRetrieval_02/evaluate_retrieval_lifelong.py', '--config', 'NewRetrieval_02/ip102_t2_retrieval.py', '--checkpoint', 'work_dirs/ip102_t2_retrieval/best_coco_Current class AP50_epoch_1.pth', '--dataset-root', '/kaggle/input/datasets/nta212/ip102-for-object-detection', '--current-task', '2', '--query-cache', 'query_cache_t2.pkl', '--gallery-cache', 'gallery_cache_t2.pkl', '--output-report', 'retrieval_lifelong_report_t2.md', '--history-file', 'history_metrics.json'], returncode=0)

## 🚀 Bước 7: Huấn luyện & Đánh giá Nhiệm vụ 3 (Task 3 - Thêm 6 lớp mới là 19 Lớp)

In [11]:
import os; os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t3_retrieval.py"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_3"]
if init_checkpoint is None:
    init_checkpoint = "work_dirs/ip102_t2_retrieval/best_coco_Current class AP50_epoch_1.pth"

prepare_config_with_checkpoint(3, init_checkpoint)

print(f"-> Bắt đầu huấn luyện Task 3...")
os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29502",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch"
]
subprocess.run(cmd, check=True)

-> Cấu hình NewRetrieval_02/ip102_t3_retrieval.py đã được cập nhật load_from = /kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t3.pth
-> Bắt đầu huấn luyện Task 3...


W0823 16:23:18.443000 132150705357952 torch/distributed/run.py:779] 
W0823 16:23:18.443000 132150705357952 torch/distributed/run.py:779] *****************************************
W0823 16:23:18.443000 132150705357952 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0823 16:23:18.443000 132150705357952 torch/distributed/run.py:779] *****************************************
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` 

08/23 16:24:23 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
08/23 16:24:23 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
08/23 16:24:23 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False

/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  atts = torch.load(att_embeddings)
/kaggle/working/OW_

08/23 16:24:25 - mmengine - INFO - Using SyncBatchNorm()
08/23 16:24:25 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(9           ) YOLOv5ParamSchedulerHook           
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
(NORMAL      ) PipelineSwitchHook                 
(NORMAL      ) OurWorkPiplineHook                 
 -------------------- 
before_train_iter:
(9           ) YOLOv5ParamSchedulerHook           
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      

...

[OurHeadRetrieval] Syncing weights to old_head_module for DwoPP distillation.[OurHeadRetrieval] Syncing weights to old_head_module for DwoPP distillation.



/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/23 16:34:06 - mmengine - INFO - Epoch(train) [1][ 50/109]  base_lr: 1.0000e-04 lr: 4.9000e-06  eta: 0:11:21  time: 11.5425  data_time: 0.0460  memory: 14207  grad_norm: nan  loss: 315.0323  loss_cls: 169.5208  loss_bbox: 61.7566  loss_dfl: 83.1145  loss_retrieval: 0.6385  loss_dwopp: 0.0020
08/23 16:43:56 - mmengine - INFO - Epoch(train) [1][100/109]  base_lr: 1.0000e-04 lr: 9.9000e-06  eta: 0:01:44  time: 11.7906  data_time: 0.0085  memory: 7383  grad_norm: 850.6185  loss: 273.2755  loss_cls: 127.2631  loss_bbox: 63.0101  loss_dfl: 82.4205  loss_retrieval: 0.5804  loss_dwopp: 0.0013
08/23 16:45:31 - mmengine - INFO - Exp name: ip102_t3_retrieval_20260823_162422
thr: 0.55
thr: 0.55
Saved collected distributions to data/IP102/mowod_distribution_sim1.pth
Saved collected distributions to data/IP102/mowod_distribution_sim1.pth
Selected 475 attributes. Saved to data/IP102/selected_att_embeddings.pth
disable log
Selected 475 attributes. Saved to data/IP102/selected_att_embeddings.pth
disa

[rank0]:[W823 16:47:57.248232528 ProcessGroupNCCL.cpp:1168] Warning: WARNING: process group has NOT been destroyed before we destruct ProcessGroupNCCL. On normal program exit, the application should call destroy_process_group to ensure that any pending NCCL operations have finished in this process. In rare cases this process can exit before this point and block the progress of another member of the process group. This constraint has always been present,  but this warning has only been added since PyTorch 2.4 (function operator())


CompletedProcess(args=['torchrun', '--nproc_per_node=2', '--master_port=29502', 'third_party/mmyolo/tools/train.py', 'NewRetrieval_02/ip102_t3_retrieval.py', '--launcher', 'pytorch'], returncode=0)

In [12]:
import os; os.environ.pop('CUDA_VISIBLE_DEVICES', None)
# Force kill any orphaned training/distributed processes to release GPU resources and locks
import subprocess
subprocess.run("pkill -f train.py", shell=True)
subprocess.run("pkill -f torchrun", shell=True)
import time
time.sleep(2)  # Give the system a moment to release GPU resources
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 3...")
best_checkpoint = "work_dirs/ip102_t3_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cmd = [
    "python", "-u",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t3_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "3",
    "--query-cache", "query_cache_t3.pkl",
    "--gallery-cache", "gallery_cache_t3.pkl",
    "--output-report", "retrieval_lifelong_report_t3.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

-> Đang thực hiện đánh giá suốt đời sau Task 3...
-> Fully patched transformers check_torch_load_is_safe across namespaces


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


      LIFELONG IMAGE RETRIEVAL EVALUATION PIPELINE      
-> Reading annotations...
-> Found 2176 query images and 2713 gallery images.
-> Extracting Query embeddings...


/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  atts = torch.load(att_embeddings)
/usr/local/lib/pyth

Loads checkpoint by local backend from path: work_dirs/ip102_t3_retrieval/best_coco_Current class AP50_epoch_1.pth
[OurHead] Dynamically resizing self.att_embeddings from torch.Size([2550, 512]) to torch.Size([475, 512]) to match checkpoint.
-> Loading CLIP model: /tmp/clip_model


CLIPModel LOAD REPORT from: /tmp/clip_model
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Query Extraction (BBox Detection):   0%|          | 0/2176 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/Tensor

-> Extracting Gallery embeddings...


Matching Queries: 100%|██████████| 2176/2176 [00:02<00:00, 872.62it/s]


-> Saved ROC Curve plot to: roc_curve_task_3.png

======================================== EVALUATION SUMMARY Task 3 ========================================
Global mAP:       0.2140
Recall@1:         0.5524
Recall@5:         0.8056
Recall@10:        0.8877
Recall@1 (Seen):  0.5767
Recall@1 (Unseen):0.7494
OOD AUROC:        0.6098
OOD FPR@TPR95:    0.7883
----------------------------------------
Plasticity:       0.1842
Forgetting (mAP): 0.0000 (0.00%)
Overall Change:   0.1842

-> Saved lifelong markdown evaluation report to: retrieval_lifelong_report_t3.md


CompletedProcess(args=['python', '-u', 'NewRetrieval_02/evaluate_retrieval_lifelong.py', '--config', 'NewRetrieval_02/ip102_t3_retrieval.py', '--checkpoint', 'work_dirs/ip102_t3_retrieval/best_coco_Current class AP50_epoch_1.pth', '--dataset-root', '/kaggle/input/datasets/nta212/ip102-for-object-detection', '--current-task', '3', '--query-cache', 'query_cache_t3.pkl', '--gallery-cache', 'gallery_cache_t3.pkl', '--output-report', 'retrieval_lifelong_report_t3.md', '--history-file', 'history_metrics.json'], returncode=0)

## 🚀 Bước 8: Huấn luyện & Đánh giá Nhiệm vụ 4 (Task 4 - Thêm 5 lớp cuối là 25 Lớp)

In [13]:
import os; os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t4_retrieval.py"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_4"]
if init_checkpoint is None:
    init_checkpoint = "work_dirs/ip102_t3_retrieval/best_coco_Current class AP50_epoch_1.pth"

prepare_config_with_checkpoint(4, init_checkpoint)

print(f"-> Bắt đầu huấn luyện Task 4...")
os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29503",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch"
]
subprocess.run(cmd, check=True)

-> Cấu hình NewRetrieval_02/ip102_t4_retrieval.py đã được cập nhật load_from = /kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t4.pth
-> Bắt đầu huấn luyện Task 4...


W0823 16:57:17.403000 138591820678272 torch/distributed/run.py:779] 
W0823 16:57:17.403000 138591820678272 torch/distributed/run.py:779] *****************************************
W0823 16:57:17.403000 138591820678272 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0823 16:57:17.403000 138591820678272 torch/distributed/run.py:779] *****************************************
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` 

08/23 16:58:24 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
08/23 16:58:24 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
08/23 16:58:24 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False

/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  atts = torch.load(att_embeddings)
/kaggle/working/OW_

08/23 16:58:25 - mmengine - INFO - Using SyncBatchNorm()
08/23 16:58:25 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(9           ) YOLOv5ParamSchedulerHook           
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
(NORMAL      ) PipelineSwitchHook                 
(NORMAL      ) OurWorkPiplineHook                 
 -------------------- 
before_train_iter:
(9           ) YOLOv5ParamSchedulerHook           
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      

...

[OurHeadRetrieval] Syncing weights to old_head_module for DwoPP distillation.
[OurHeadRetrieval] Syncing weights to old_head_module for DwoPP distillation.


/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/23 17:09:16 - mmengine - INFO - Epoch(train) [1][ 50/181]  base_lr: 1.0000e-04 lr: 4.9000e-06  eta: 0:28:12  time: 12.9161  data_time: 0.0699  memory: 14232  grad_norm: nan  loss: 324.1071  loss_cls: 185.0271  loss_bbox: 57.8454  loss_dfl: 80.5865  loss_retrieval: 0.6465  loss_dwopp: 0.0016
08/23 17:19:50 - mmengine - INFO - Epoch(train) [1][100/181]  base_lr: 1.0000e-04 lr: 9.9000e-06  eta: 0:17:16  time: 12.6803  data_time: 0.0080  memory: 7484  grad_norm: 882.4792  loss: 264.4395  loss_cls: 132.2323  loss_bbox: 54.8707  loss_dfl: 76.7639  loss_retrieval: 0.5716  loss_dwopp: 0.0011
08/23 17:30:43 - mmengine - INFO - Epoch(train) [1][150/181]  base_lr: 1.0000e-04 lr: 1.4900e-05  eta: 0:06:39  time: 13.0649  data_time: 0.0079  memory: 7485  grad_norm: 665.4504  loss: 236.2041  loss_cls: 99.9840  loss_bbox: 57.3120  loss_dfl: 78.4188  loss_retrieval: 0.4885  loss_dwopp: 0.0008
08/23 17:37:24 - mmengine - INFO - Exp name: ip102_t4_retrieval_20260823_165823
thr: 0.55
thr: 0.55
Saved co

[rank0]:[W823 17:39:51.208435826 ProcessGroupNCCL.cpp:1168] Warning: WARNING: process group has NOT been destroyed before we destruct ProcessGroupNCCL. On normal program exit, the application should call destroy_process_group to ensure that any pending NCCL operations have finished in this process. In rare cases this process can exit before this point and block the progress of another member of the process group. This constraint has always been present,  but this warning has only been added since PyTorch 2.4 (function operator())


CompletedProcess(args=['torchrun', '--nproc_per_node=2', '--master_port=29503', 'third_party/mmyolo/tools/train.py', 'NewRetrieval_02/ip102_t4_retrieval.py', '--launcher', 'pytorch'], returncode=0)

In [14]:
import os; os.environ.pop('CUDA_VISIBLE_DEVICES', None)
# Force kill any orphaned training/distributed processes to release GPU resources and locks
import subprocess
subprocess.run("pkill -f train.py", shell=True)
subprocess.run("pkill -f torchrun", shell=True)
import time
time.sleep(2)  # Give the system a moment to release GPU resources
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 4...")
best_checkpoint = "work_dirs/ip102_t4_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cmd = [
    "python", "-u",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t4_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "4",
    "--query-cache", "query_cache_t4.pkl",
    "--gallery-cache", "gallery_cache_t4.pkl",
    "--output-report", "retrieval_lifelong_report_t4.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

-> Đang thực hiện đánh giá suốt đời sau Task 4...
-> Fully patched transformers check_torch_load_is_safe across namespaces


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


      LIFELONG IMAGE RETRIEVAL EVALUATION PIPELINE      
-> Reading annotations...
-> Found 2176 query images and 2713 gallery images.
-> Extracting Query embeddings...


/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  atts = torch.load(att_embeddings)
/usr/local/lib/pyth

Loads checkpoint by local backend from path: work_dirs/ip102_t4_retrieval/best_coco_Current class AP50_epoch_1.pth
[OurHead] Dynamically resizing self.att_embeddings from torch.Size([2550, 512]) to torch.Size([625, 512]) to match checkpoint.
-> Loading CLIP model: /tmp/clip_model


CLIPModel LOAD REPORT from: /tmp/clip_model
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Query Extraction (BBox Detection):   0%|          | 0/2176 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/Tensor

-> Extracting Gallery embeddings...


Matching Queries: 100%|██████████| 2176/2176 [00:02<00:00, 833.74it/s]



======================================== EVALUATION SUMMARY Task 4 ========================================
Global mAP:       0.2179
Recall@1:         0.5438
Recall@5:         0.7868
Recall@10:        0.8762
Recall@1 (Seen):  0.6407
Recall@1 (Unseen):None
OOD AUROC:        None
OOD FPR@TPR95:    None
----------------------------------------
Plasticity:       0.4240
Forgetting (mAP): 0.0014 (0.14%)
Overall Change:   0.4227

-> Saved lifelong markdown evaluation report to: retrieval_lifelong_report_t4.md


CompletedProcess(args=['python', '-u', 'NewRetrieval_02/evaluate_retrieval_lifelong.py', '--config', 'NewRetrieval_02/ip102_t4_retrieval.py', '--checkpoint', 'work_dirs/ip102_t4_retrieval/best_coco_Current class AP50_epoch_1.pth', '--dataset-root', '/kaggle/input/datasets/nta212/ip102-for-object-detection', '--current-task', '4', '--query-cache', 'query_cache_t4.pkl', '--gallery-cache', 'gallery_cache_t4.pkl', '--output-report', 'retrieval_lifelong_report_t4.md', '--history-file', 'history_metrics.json'], returncode=0)

## 📊 Bước 9: Tổng hợp và hiển thị Ma trận học trọn đời (Lifelong Performance Matrix)
Hiển thị chi tiết bảng so sánh chất lượng truy xuất qua các pha huấn luyện để theo dõi mức độ ổn định của thuật toán chưng cất DwoPP.

In [15]:
import json
import pandas as pd
from IPython.display import display, Markdown

if os.path.exists("history_metrics.json"):
    with open("history_metrics.json", "r") as f:
        history = json.load(f)
        
    rows = []
    for stage, metrics in sorted(history.items()):
        rows.append({
            "Giai đoạn Đánh giá": stage.upper().replace("_", " "),
            "T1 mAP (7 lớp đầu)": f"{metrics.get('T1', {}).get('mAP', 0.0):.4f}",
            "T2 mAP (lớp 8-13)": f"{metrics.get('T2', {}).get('mAP', 0.0):.4f}",
            "T3 mAP (lớp 14-19)": f"{metrics.get('T3', {}).get('mAP', 0.0):.4f}",
            "T4 mAP (lớp 20-25)": f"{metrics.get('T4', {}).get('mAP', 0.0):.4f}",
        })
        
    df = pd.DataFrame(rows)
    display(Markdown("### 📈 Ma trận kết quả mAP học trọn đời:"))
    display(df)
    
    # Tính Forgetting & Plasticity cuối cùng sau Task 4
    if "task_4" in history and "task_1" in history:
        ap_t1_t1 = history["task_1"]["T1"]["mAP"]
        ap_t1_t4 = history["task_4"]["T1"]["mAP"]
        ap_t2_t2 = history["task_2"]["T2"]["mAP"]
        ap_t2_t4 = history["task_4"]["T2"]["mAP"]
        ap_t3_t3 = history["task_3"]["T3"]["mAP"]
        ap_t3_t4 = history["task_4"]["T3"]["mAP"]
        
        f1 = max(0.0, ap_t1_t1 - ap_t1_t4)
        f2 = max(0.0, ap_t2_t2 - ap_t2_t4)
        f3 = max(0.0, ap_t3_t3 - ap_t3_t4)
        forgetting = (f1 + f2 + f3) / 3.0
        plasticity = history["task_4"]["T4"]["mAP"]
        overall = plasticity - forgetting
        
        summary_md = f"""
### 📊 Chỉ số học trọn đời tích hợp (sau Task 4):
*   **Plasticity (Khả năng tiếp thu mới):** `{plasticity:.4f}`
*   **Forgetting (Độ quên lãng trung bình):** `{forgetting:.4f} ({forgetting*100:.2f}%)`
*   **Overall Change (Độ ổn định hệ thống):** `{overall:.4f}`
"""
        display(Markdown(summary_md))
else:
    print("-> File history_metrics.json không tồn tại. Hãy chạy đầy đủ các tác vụ huấn luyện và đánh giá trước.")

### 📈 Ma trận kết quả mAP học trọn đời:

,Giai đoạn Đánh giá,T1 mAP (7 lớp đầu),T2 mAP (lớp 8-13),T3 mAP (lớp 14-19),T4 mAP (lớp 20-25)
0,TASK 1,0.2857,0.1273,0.1835,0.4183
1,TASK 2,0.2868,0.1271,0.1838,0.4181
2,TASK 3,0.2863,0.1271,0.1842,0.4176
3,TASK 4,0.3035,0.1230,0.1858,0.4240



### 📊 Chỉ số học trọn đời tích hợp (sau Task 4):
*   **Plasticity (Khả năng tiếp thu mới):** `0.4240`
*   **Forgetting (Độ quên lãng trung bình):** `0.0014 (0.14%)`
*   **Overall Change (Độ ổn định hệ thống):** `0.4227`


In [16]:
# === Cell 25 ===
import zipfile
import glob
import os

zip_name = "/kaggle/working/retrieval_caches.zip"
pkl_files = glob.glob("*.pkl")
report_files = glob.glob("*.md")
png_files = glob.glob("*.png")
json_files = glob.glob("*.json")

files_to_zip = pkl_files + report_files + png_files + json_files
if files_to_zip:
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for f in files_to_zip:
            if os.path.exists(f) and f != "log.txt":
                zipf.write(f, os.path.basename(f))
    print(f"-> Đã đóng gói thành công {len(files_to_zip)} file vào {zip_name}")
else:
    print("-> Không tìm thấy file cache hoặc báo cáo nào để đóng gói.")


-> Đã đóng gói thành công 18 file vào /kaggle/working/retrieval_caches.zip
